In [1]:
import pandas as pd
import numpy as np
import gc

import torch
import faiss

from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_parquet('new_df.parquet')

df = df.sort_values("date").reset_index(drop=True)

content_train = df[
    df["date"] < pd.Timestamp("2023-07-14")
].copy()

val = df[
    (df["date"] >= pd.Timestamp("2023-07-14")) &
    (df["date"] < pd.Timestamp("2023-08-14"))
].copy()

In [3]:
def hit_at_k(recom, true_ans, k):
    recom = recom[:k]
    true_ans = set(true_ans)

    if len(true_ans) == 0:
        return 0

    return int(
        len(set(recom) & true_ans) > 0
    )


def recall_at_k(recom, true_ans, k):
    recom = recom[:k]
    true_ans = set(true_ans)

    if len(true_ans) == 0:
        return 0

    hits = len(set(recom) & true_ans)

    return hits / len(true_ans)

In [4]:
meta = pd.read_parquet(
    "/home/user/mle/data/res/meta.parquet",
    columns=[
        "parent_asin",
        "title",
        "main_category",
        "categories",
        "details"])

meta = meta.drop_duplicates("parent_asin").reset_index(drop=True)


print("Items:", len(meta))

meta.isna().mean().sort_values(ascending=False)

Items: 5873289


categories       0.131052
parent_asin      0.000000
title            0.000000
main_category    0.000000
details          0.000000
dtype: float64

In [5]:
meta

,parent_asin,title,main_category,categories,details
0,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",All Beauty,None,Package Dimensions: 7.1 x 5.5 x 3 inches; 2.38...
1,B076WQZGPM,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,All Beauty,None,Item Form: Powder; Skin Type: Acne Prone; Bran...
2,B000B658RI,Eye Patch Black Adult with Tie Band (6 Per Pack),All Beauty,None,Manufacturer: Levine Health Products
3,B088FKY3VD,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",All Beauty,None,Brand: Cherioll; Item Form: Powder; Finish Typ...
4,B07NGFDN6G,Precision Plunger Bars for Cartridge Grips – 9...,All Beauty,None,UPC: 644287689178
...,...,...,...,...,...
5873284,B00KW21CC6,Serotonin Molecular Model with Smiley Faces Vi...,Computers,"[Electronics, Computers & Accessories, Laptop ...",Brand: Pandora Stickers Arts and Hobbies; Room...
5873285,B07WTBH65P,Super Uncle Birthday Party Backdrop for Photog...,Camera & Photo,"[Electronics, Camera & Photo, Lighting & Studi...","Product Dimensions: 108""L x 72""W; Item Weight:..."
5873286,B003NUIU9M,"Wintec FileMate Pro USB Flash Drive, 3FMUSB32G...",Computers,"[Electronics, Computers & Accessories, Data St...",Product Dimensions: 0.78 x 0.31 x 2.75 inches;...
5873287,B091JWCSG5,"FYY 12-13.3"" Laptop Sleeve Case Bag, PU Leathe...",Computers,"[Electronics, Computers & Accessories, Laptop ...",Standing screen display size: 12.3 Inches; Bra...


In [6]:
def to_text(x):
    if x is None:
        return ""

    if isinstance(x, dict):
        return " ".join(
            f"{key}: {value}"
            for key, value in x.items())

    if isinstance(x, (list, tuple, np.ndarray)):
        return " ".join(map(str, x))

    return str(x)

In [7]:
sample_meta = meta[
    meta["title"].notna() &
    meta["main_category"].notna() &
    meta["categories"].notna() &
    meta["details"].notna()
].sample(
    n=min(5000, meta["details"].notna().sum()),
    random_state=42).reset_index(drop=True)

In [8]:
sample_meta["title_text"] = sample_meta["title"].fillna("")

sample_meta["without_details"] = (
    sample_meta["title"].fillna("")
    + ". "
    + sample_meta["main_category"].fillna("")
    + ". "
    + sample_meta["categories"].apply(to_text))

sample_meta["full_text"] = (
    sample_meta["without_details"]
    + ". "
    + sample_meta["details"].apply(to_text))

In [9]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
model.max_seq_length = 256

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
text_results = []

for mode, text_col in [
    ("title", "title_text"),
    ("without_details", "without_details"),
    ("full", "full_text")
]:

    sample_embeddings = model.encode(
        sample_meta[text_col].tolist(),
        device="cuda",
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype(np.float32)

    sample_index = faiss.IndexFlatIP(
        sample_embeddings.shape[1]
    )

    sample_index.add(np.ascontiguousarray(sample_embeddings))

    scores, neighbors = sample_index.search(np.ascontiguousarray(sample_embeddings),11)

    neighbors = neighbors[:, 1:]

    categories = (
        sample_meta["main_category"]
        .to_numpy()
    )

    same_category = (
        categories[neighbors]
        == categories[:, None]
    ).mean()

    text_results.append({
        "text": mode,
        "same_category_at_10": same_category
    })

In [11]:
text_quality = pd.DataFrame(
    text_results
)

text_quality

,text,same_category_at_10
0,title,0.74454
1,without_details,0.85106
2,full,0.87306


In [12]:
meta["text"] = (
    meta["title"].fillna("")
    + ". "
    + meta["main_category"].fillna("")
    + ". "
    + meta["categories"].apply(to_text)
)

In [13]:
meta['text']

0          Howard LC0008 Leather Conditioner, 8-Ounce (4-...
1          Yes to Tomatoes Detoxifying Charcoal Cleanser ...
2          Eye Patch Black Adult with Tie Band (6 Per Pac...
3          Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...
4          Precision Plunger Bars for Cartridge Grips – 9...
                                 ...                        
5873284    Serotonin Molecular Model with Smiley Faces Vi...
5873285    Super Uncle Birthday Party Backdrop for Photog...
5873286    Wintec FileMate Pro USB Flash Drive, 3FMUSB32G...
5873287    FYY 12-13.3" Laptop Sleeve Case Bag, PU Leathe...
5873288    4MP Full Time Color Night Vision POE IP Camera...
Name: text, Length: 5873289, dtype: str

In [14]:
embeddings = model.encode(
    meta["text"].tolist(),
    device="cuda",
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float32)

print("Embeddings:", embeddings.shape)

Batches:   0%|          | 0/183541 [00:00<?, ?it/s]

Embeddings: (5873289, 384)


In [16]:
item_index = meta[["parent_asin"]].copy()

item_index["item_idx"] = np.arange(len(item_index))

asin_to_idx = item_index.set_index("parent_asin")["item_idx"]

idx_to_asin = item_index["parent_asin"].to_numpy()

In [17]:
embedding_ln = embeddings.shape[1]

quantizer = faiss.IndexFlatIP(embedding_ln)

index = faiss.IndexIVFFlat(
    quantizer,
    embedding_ln,
    6000,
    faiss.METRIC_INNER_PRODUCT
)

index.train(embeddings)

index.add(embeddings)

index.nprobe = 50

print("Items in index:", index.ntotal)

Items in index: 5873289


In [18]:
content_train = content_train.sort_values("date").drop_duplicates(["user_id", "parent_asin"],keep="last")

content_train = content_train[content_train["rating"] >= 4].copy()

content_train = content_train[
        content_train["parent_asin"].isin(
        asin_to_idx.index)].copy()

content_train["item_idx"] = content_train["parent_asin"].map(asin_to_idx).astype(int)


print("Positive interactions:", len(content_train))
print("Users:", content_train["user_id"].nunique())

Positive interactions: 7874996
Users: 4744979


In [19]:
user_history = content_train.groupby("user_id")["item_idx"].agg(list)

In [20]:
user_seen = content_train.groupby("user_id")["parent_asin"].agg(set)

In [21]:
val_pos = val[val["rating"] >= 4].copy()

In [22]:
train_seen = content_train[["user_id", "parent_asin"]].drop_duplicates()
content_target = val_pos.merge(train_seen.assign(seen=1), on=["user_id", "parent_asin"], how="left")
content_target = content_target[content_target["seen"].isna()]

In [23]:
content_target = content_target[content_target["user_id"].isin(user_history.index)]
content_target = content_target[content_target["parent_asin"].isin(asin_to_idx.index)]
content_targets = content_target.groupby("user_id")["parent_asin"].agg(set)

print("Users for evaluation:", len(content_targets))

Users for evaluation: 7586


In [24]:
max_recommendations = 1000
neighbors_per_item = 1000

recommendations = {}

for user_id in content_targets.index:

    history_idx = user_history[user_id]
    queries = embeddings[history_idx]
    

    scores, neighbors = index.search(queries, neighbors_per_item)

    seen = user_seen[user_id]

    candidates = {}

    for idx, score in zip(neighbors.ravel(), scores.ravel()):

        if idx == -1:
            continue

        asin = idx_to_asin[idx]

        if asin in seen:
            continue

        if asin not in candidates or score > candidates[asin]:
            candidates[asin] = score

    recommendations[user_id] = [
        asin
        for asin, score in sorted(
            candidates.items(),
            key=lambda x: x[1],
            reverse=True
        )[:max_recommendations]]

In [25]:
k_values = [10, 50, 100, 300, 1000]

rows = []

for k in k_values:

    hit_scores = []
    recall_scores = []

    for user_id, recs in recommendations.items():

        true_items = content_targets[user_id]

        hit_scores.append(hit_at_k(recs, true_items,k))

        recall_scores.append(recall_at_k(recs, true_items, k))

    rows.append({
        "k": k,
        "hit_at_k": np.mean(hit_scores),
        "recall_at_k": np.mean(recall_scores)
    })

content_quality = pd.DataFrame(rows)

content_quality

,k,hit_at_k,recall_at_k
0,10,0.020828,0.017957
1,50,0.035724,0.031474
2,100,0.040469,0.035453
3,300,0.054179,0.047178
4,1000,0.075534,0.065980


In [ ]:
np.save("content_embeddings.npy", embeddings)
item_index.to_parquet("content_item_index.parquet", index=False)
faiss.write_index(index, "content_faiss.index")